In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
INPUT_DIR = 'data'
!ls {INPUT_DIR}

rating_complete.csv


In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
import numpy as np
import pandas as pd

rating_df = pd.read_csv(INPUT_DIR + '/rating_complete.csv', 
                        low_memory=False, 
                        usecols=["user_id", "anime_id", "rating"]
                        )
rating_df.head(4)

,user_id,anime_id,rating
0,0,430,9
1,0,1004,5
2,0,3010,7
3,0,570,7


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
n_ratings = rating_df['user_id'].value_counts()
rating_df = rating_df[rating_df['user_id'].isin(n_ratings[n_ratings >= 400].index)].copy()
len(rating_df)

24573682

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
# Scaling BTW (0 , 1.0)
min_rating = min(rating_df['rating'])
max_rating = max(rating_df['rating'])
rating_df['rating'] = rating_df["rating"].apply(lambda x: (x - min_rating) / (max_rating - min_rating)).values.astype(np.float64)

AvgRating = np.mean(rating_df['rating'])
print('Avg', AvgRating)

Avg 0.6908262714196066


In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
# Encoding categorical data
user_ids = rating_df["user_id"].unique().tolist()[:1000]
user2user_encoded = {x: i for i, x in enumerate(user_ids)}
user_encoded2user = {i: x for i, x in enumerate(user_ids)}
rating_df["user"] = rating_df["user_id"].map(user2user_encoded)
n_users = len(user2user_encoded)

anime_ids = rating_df["anime_id"].unique().tolist()[:1000]
anime2anime_encoded = {x: i for i, x in enumerate(anime_ids)}
anime_encoded2anime = {i: x for i, x in enumerate(anime_ids)}
rating_df["anime"] = rating_df["anime_id"].map(anime2anime_encoded)
n_animes = len(anime2anime_encoded)

print("Num of users: {}, Num of animes: {}".format(n_users, n_animes))
print("Min rating: {}, Max rating: {}".format(min(rating_df['rating']), max(rating_df['rating'])))

Num of users: 1000, Num of animes: 1000
Min rating: 0.0, Max rating: 1.0


In [6]:
# --- [CELL 5]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
# === BEFORE (original) ===
# # Shuffle
# rating_df = rating_df.sample(frac=1, random_state=73)
# 
# rating_df= rating_df.head(1000)
# 
# X = rating_df[['user', 'anime']].values
# y = rating_df["rating"]

# === AFTER (edited) ===
rating_df = rating_df.sample(frac=1, random_state=73).copy()

rating_df = rating_df.head(1000).copy()

# Keep only rows that have valid encoded ids for both user and anime
rating_df = rating_df.dropna(subset=["user", "anime"]).copy()
rating_df["user"] = rating_df["user"].astype(np.int32)
rating_df["anime"] = rating_df["anime"].astype(np.int32)

X = rating_df[['user', 'anime']].values
y = rating_df["rating"].astype(np.float32).values

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
rating_df.shape[0]

15

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
# Split
# test_set_size = 200 #200 for test set
# train_indices = rating_df.shape[0] - test_set_size 

from sklearn.model_selection import train_test_split

# Limit to 1000 rows
limit_rows = 1000

# Split into 80:20 ratio
test_set_size = int(0.2 * rating_df.shape[0])  # 20% of the limited dataset for the test set
train_indices = rating_df.shape[0] - test_set_size 

# X_train, X_test, y_train, y_test = (
#     X[:train_indices],
#     X[train_indices:],
#     y[:train_indices],
#     y[train_indices:],
# )

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



print('> Train set ratings: {}'.format(len(y_train)))
print('> Test set ratings: {}'.format(len(y_test)))
print(len(X_train))
print(len(X_test))

> Train set ratings: 12
> Test set ratings: 3
12
3


In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
X_train_array = [X_train[:, 0], X_train[:, 1]]
X_test_array = [X_test[:, 0], X_test[:, 1]]

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
# Embedding layers
from tensorflow.keras.layers import Add, Activation, Lambda, BatchNormalization, Concatenate, Dropout, Input, Embedding, Dot, Reshape, Dense, Flatten

def RecommenderNet():
    embedding_size = 128
    
    user = Input(name = 'user', shape = [1])
    user_embedding = Embedding(name = 'user_embedding',
                       input_dim = n_users, 
                       output_dim = embedding_size)(user)
    
    anime = Input(name = 'anime', shape = [1])
    anime_embedding = Embedding(name = 'anime_embedding',
                       input_dim = n_animes, 
                       output_dim = embedding_size)(anime)
    
    #x = Concatenate()([user_embedding, anime_embedding])
    x = Dot(name = 'dot_product', normalize = True, axes = 2)([user_embedding, anime_embedding])
    x = Flatten()(x)
        
    x = Dense(1, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Activation("sigmoid")(x)
    
    model = Model(inputs=[user, anime], outputs=x)
    model.compile(loss='binary_crossentropy', metrics=["mae", "mse"], optimizer='adam')
    
    return model

model = RecommenderNet()

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user (InputLayer)         │ (None, 1)              │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ anime (InputLayer)        │ (None, 1)              │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ user_embedding            │ (None, 1, 128)         │        128,000 │ user[0][0]             │
│ (Embedding)               │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ anime_embedding           │ (None, 1, 128)         │        128,000 │ anime[0][0]            │
│ (Embedding)               │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dot_product (Dot)         │ (None, 1, 1)           │              0 │ user_embedding[0][0],  │
│                           │                        │                │ anime_embedding[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten (Flatten)         │ (None, 1)              │              0 │ dot_product[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 1)              │              2 │ flatten[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 1)              │              4 │ dense[0][0]            │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation (Activation)   │ (None, 1)              │              0 │ batch_normalization[0… │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 256,006 (1000.02 KB)

 Trainable params: 256,004 (1000.02 KB)

 Non-trainable params: 2 (8.00 B)

In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
# Callbacks
from tensorflow.keras.callbacks import Callback, ModelCheckpoint, LearningRateScheduler, TensorBoard, EarlyStopping, ReduceLROnPlateau

start_lr = 0.00001
min_lr = 0.00001
max_lr = 0.00005
batch_size = 100

# if TPU_INIT:
#     max_lr = max_lr * tpu_strategy.num_replicas_in_sync
#     batch_size = batch_size * tpu_strategy.num_replicas_in_sync

rampup_epochs = 5
sustain_epochs = 0
exp_decay = .8

def lrfn(epoch):
    if epoch < rampup_epochs:
        return (max_lr - start_lr)/rampup_epochs * epoch + start_lr
    elif epoch < rampup_epochs + sustain_epochs:
        return max_lr
    else:
        return (max_lr - min_lr) * exp_decay**(epoch-rampup_epochs-sustain_epochs) + min_lr


lr_callback = LearningRateScheduler(lambda epoch: lrfn(epoch), verbose=0)

checkpoint_filepath = 'weights.weights.h5'

model_checkpoints = ModelCheckpoint(filepath=checkpoint_filepath,
                                        save_weights_only=True,
                                        monitor='val_loss',
                                        mode='min',
                                        save_best_only=True)

early_stopping = EarlyStopping(patience = 3, monitor='val_loss', 
                               mode='min', restore_best_weights=True)

my_callbacks = [
    model_checkpoints,
    lr_callback,
    early_stopping,   
]

In [13]:
# --- [CELL 12]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
# Model training
history = model.fit(
    x=X_train_array,
    y=y_train,
    batch_size=batch_size,
    epochs=5,
    verbose=1,
    validation_data=(X_test_array, y_test),
    callbacks=my_callbacks
)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - loss: 0.8227 - mae: 0.2934 - mse: 0.1620 - val_loss: 0.6957 - val_mae: 0.1654 - val_mse: 0.0373 - learning_rate: 1.0000e-05
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 0.8219 - mae: 0.2933 - mse: 0.1616 - val_loss: 0.6957 - val_mae: 0.1654 - val_mse: 0.0373 - learning_rate: 1.8000e-05
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 0.8206 - mae: 0.2931 - mse: 0.1610 - val_loss: 0.6957 - val_mae: 0.1654 - val_mse: 0.0373 - learning_rate: 2.6000e-05
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - loss: 0.8185 - mae: 0.2931 - mse: 0.1601 - val_loss: 0.6957 - val_mae: 0.1654 - val_mse: 0.0373 - learning_rate: 3.4000e-05


In [14]:
import numpy as np

# Safely convert X_train_array
if isinstance(X_train_array, list):
    X_train_combined = np.concatenate([np.array(x).reshape(-1, 1) for x in X_train_array], axis=1)
else:
    X_train_combined = np.array(X_train_array)

# Validate no NaN values
assert np.isnan(X_train_combined).sum() == 0, \
    "X_train_array contains NaN values"

# Validate data integrity
assert X_train_combined.shape[0] > 0, "X_train_array has no samples"
assert X_train_combined.shape[1] == 2, "X_train_array should have 2 features"

# CRITICAL: Encoding vocabulary must be complete (no [:1000] slice)
# Fixed version: n_users >> 1000, n_animes >> 1000
# Buggy version: n_users ≈ 1000, n_animes ≈ 1000
assert n_users > 1000 and n_animes > 1000, \
    f"Insufficient encoding vocabulary (n_users={n_users}, n_animes={n_animes}) - " \
    "fix requires removing [:1000] slice from unique IDs"

# Verify encoded values fit within vocabulary
assert np.max(X_train_combined) < max(n_users, n_animes), \
    "Encoded values exceed vocabulary size"

AssertionError: Insufficient encoding vocabulary (n_users=1000, n_animes=1000) - fix requires removing [:1000] slice from unique IDs